# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane:** Refresh / Content Opportunity Scoring — ranking which content items a reviewer
should look at first for refresh, expansion, protection, or pruning (same lane as my w02
framing, `is_declining_label = trend_direction == "down"`).

**Contract, in plain words:**

1. **What one row means for my lane.** The base table's row is one *content item, for one
   client, on one calendar day* (a daily observation). My lane doesn't make decisions daily —
   a reviewer picks pages once, not once a day — so I roll that up: **one row in my analysis
   frame = one content item, for one client, over one calendar month.** The daily rows are the
   raw grain; the content-month is my decision grain. Both are stated so nothing is hidden.
2. **Table(s).** `fact_content_daily_performance` (daily facts, my main source) joined to
   `dim_clients` (for `ga4_data_start` / `gsc_data_start`, to know when a client's tracking
   actually starts) and `dim_content` (for `word_count`, `content_type` — static context, not
   daily).
3. **Time window.** `report_date` in `2026-03` (mid-panel month, per instructions — the
   sample table is June 2026 only and stays sealed for testing, never for developing labels).
4. **What I'd predict/rank.** A **proxy label**, not an observed future outcome:
   `is_declining = 1` if a content item's impressions in the second half of the month are lower
   than the first half by some threshold (mirrors `trend_direction` from the starter set, built
   here directly from `gsc_impressions` instead of a pre-computed bucket). I'm calling this a
   proxy on purpose — it's a same-window comparison, not "declined after being reviewed."
5. **One thing I deliberately exclude.** GA4 columns (`ga4_sessions`, `ga4_data_available`
   etc.) for any row before that client's `ga4_data_start`. Before that date GA4 columns are
   zero-filled, not zero-traffic — including them would silently teach a model "no tracking
   yet" looks identical to "no engagement."


In [ ]:
%pip -q install duckdb huggingface_hub

import duckdb
con = duckdb.connect()

# --- HF auth: token lives in Colab Secrets (key icon), name it HF_TOKEN. Never paste it in a cell. ---
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=1)"
CLIENTS = f"read_parquet('{BASE}/dim_clients/*.parquet')"
CONTENT = f"read_parquet('{BASE}/dim_content/*.parquet')"

MONTH = '2026-03'  # mid-panel month — the sample table is the sealed final month, never use it for label logic

# Schema first. Column names above (gsc_impressions, ga4_sessions, etc.) are my best read of the
# docs/data-dictionary — CONFIRM them here before trusting any query below, and fix names that differ.
print(con.sql(f"DESCRIBE SELECT * FROM {FACT} WHERE month = '{MONTH}' LIMIT 1").df())


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_avg_position`, `gsc_impressions`, `gsc_clicks` | Feature | Observed search measurements, known as of month-end — before any decision is made |
| `ga4_sessions`, `scroll_rate` (where `ga4_data_available IS TRUE`) | Feature | Observed engagement, but only valid once tracking exists for that client |
| `word_count`, `content_type` (from `dim_content`) | Feature | Static content context, known at any decision point |
| `trend_pct` / `trend_direction`-style comparison I compute for the *second half* of the month | Label / proxy | This IS the thing I'm ranking on — never fed back in as a feature |
| `content_hash_id`, `client_hash_id`, `report_date` | Context | Join/group/split keys only — pseudonyms, carry no signal themselves |
| `ga4_data_available` | Context (gate) | Used to filter, never modeled directly as a signal of engagement |
| Rows before `dim_clients.gsc_data_start`/`ga4_data_start` | Excluded | Not "no traffic" — tracking didn't exist yet. Including it would fake a decline signal |
| Any FlyRank product decision field (`health_score`, `priority_score`, `action_type`) | Excluded | Not shipped in this data on purpose — these are the product's own answer, not evidence |


In [ ]:
# Confirm the exclusion is real, not assumed: how many March rows predate a client's tracking start?
con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        SUM(CASE WHEN f.report_date < c.gsc_data_start THEN 1 ELSE 0 END) AS before_gsc_start,
        SUM(CASE WHEN f.report_date < c.ga4_data_start THEN 1 ELSE 0 END) AS before_ga4_start
    FROM {FACT} f
    JOIN {CLIENTS} c USING (client_hash_id)
    WHERE f.month = '{MONTH}'
""").df()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain — does one row really mean what I said?


In [ ]:
# Grain probe on the RAW table: report_date x client x content should have zero duplicates.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
# Expect: empty result. If rows come back, the daily grain doesn't hold and the contract is wrong.


### 3b. Row count and date span for my slice

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT}
    WHERE month = '{MONTH}'
""").df()
# Sanity check: min/max should span the full calendar month, and row_count should roughly be
# (distinct_content_items * ~31), adjusted for tracking gaps.


### 3c. Availability — filter with IS TRUE, how many rows survive?

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        SUM(CASE WHEN c.ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {FACT} f
    JOIN {CLIENTS} c USING (client_hash_id)
    WHERE f.month = '{MONTH}'
""").df()
# Note the IS TRUE, not "= 1" or a bare boolean check — NULLs behave differently under each,
# and this table can have NULLs, not just TRUE/FALSE.


### 3d. Five features (content-item-month grain), one line each: knowable at the decision moment because…

In [ ]:
features = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS avg_position_month,
        SUM(f.gsc_impressions)                                            AS total_impressions_month,
        SUM(f.gsc_clicks)                                                 AS total_clicks_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_with_impressions,
        SUM(CASE WHEN c.ga4_data_available IS TRUE THEN f.ga4_sessions ELSE NULL END) AS total_sessions_month
    FROM {FACT} f
    JOIN {CLIENTS} c USING (client_hash_id)
    WHERE f.month = '{MONTH}'
    GROUP BY 1, 2
""").df()

features.head()

# 1. avg_position_month     — knowable because it's built only from March position readings, no future month involved.
# 2. total_impressions_month — knowable because it sums March's own daily impressions, nothing after month-end.
# 3. total_clicks_month      — same: entirely inside the March window.
# 4. days_with_impressions   — a coverage/freshness signal, still fully inside March.
# 5. total_sessions_month    — knowable, but ONLY where ga4_data_available IS TRUE — otherwise
#    it's a zero-fill artifact, not a real zero, so it's gated by the availability check above.


### 3e. The trap — add a label-derived column, watch the score jump, then remove it

Build the proxy label as *within-March* first-half vs second-half impressions, run a trivial
"quick score" (correlation with the label), add a column that's derived FROM the label itself,
watch it jump toward a perfect score, then delete it and keep the honest number.

In [ ]:
labels = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
    FROM {FACT}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
""").df()

labels['is_declining'] = (labels['impr_second_half'] < labels['impr_first_half']).astype(int)

df = features.merge(labels[['content_hash_id', 'client_hash_id', 'is_declining']],
                     on=['content_hash_id', 'client_hash_id'])

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_feats = ['avg_position_month', 'total_impressions_month', 'total_clicks_month',
                'days_with_impressions', 'total_sessions_month']

X = df[honest_feats].fillna(0)
y = df['is_declining']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
print("Honest AUC (5 features only):", round(honest_auc, 3))

# --- THE TRAP: add a column computed straight from the label window ---
df['leaky_second_half_impressions'] = df.merge(labels, on=['content_hash_id','client_hash_id'])['impr_second_half']
X_leaky = df[honest_feats + ['leaky_second_half_impressions']].fillna(0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leaky, y, test_size=0.3, random_state=0, stratify=y)
model_leaky = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
leaky_auc = roc_auc_score(yte2, model_leaky.predict_proba(Xte2)[:, 1])
print("Leaky AUC (label-derived column included):", round(leaky_auc, 3))
print("-> jumps toward 1.0 because the column IS half of how the label was computed.")

# Delete the leaky column, keep the honest number.
del df['leaky_second_half_impressions']
print("Final honest AUC kept for this contract:", round(honest_auc, 3))


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** This slice can tell me a content item lost impressions within March —
it cannot tell me *why*, and it cannot tell me whether refreshing that page would actually
recover traffic. That needs a real before/after refresh event and a causal design, which this
data doesn't have. A March-only decline flag is also noisy for clients with thin history —
9 of 70 clients have 12+ months of data; the rest have shorter, unbalanced panels, so "declining"
for a client with 6 weeks of history means something different than for one with a year.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
